# Week 2 — Competitive Analysis

Loads the Week 2 data files, validates coverage, summarizes patent intelligence and R&D scale (real headcount + patents), and charts AI-ML hiring share once a dated snapshot is filled.

All inputs are public-sourced. Patent numbers and headcounts are real; AI-ML hiring counts are a blank dated-snapshot template (not fabricated).

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('..')/'data'/'week02'
pos = pd.read_csv(base/'peers_positioning.csv')
pat = pd.read_excel(base/'patents.xlsx', sheet_name='patent_activity')
rd  = pd.read_excel(base/'rd_scale_and_hiring.xlsx', sheet_name='rd_scale_real')
tmpl= pd.read_excel(base/'rd_scale_and_hiring.xlsx', sheet_name='hiring_snapshot_TEMPLATE')
print('peers:', len(pos), '| patents:', len(pat), '| rd_scale:', len(rd))

## 1. Coverage check — 3 peers per vertical, 15 total

In [ ]:
print(pos.groupby('vertical')['company'].count())
assert len(pos)==15
print('\nAnchors covered:', sorted(pos['anchor_ingen_product'].unique()))

## 2. Patent intelligence — peers with a concrete patent number located

In [ ]:
import re
col=[c for c in pat.columns if 'Specific' in c][0]
pat['has_number']=pat[col].apply(lambda s: bool(re.search(r'(US ?\d|D\d|US\d)', str(s))))
print(pat[['Company','has_number']].to_string(index=False))
print('\nConcrete patent number located for', int(pat['has_number'].sum()), 'of', len(pat), 'peers')

## 3. R&D scale — real headcount + IP read (verifiable)

In [ ]:
ecol=[c for c in rd.columns if 'Employees' in c][0]
print(rd[['Company',ecol]].to_string(index=False))

## 4. Hiring AI-ML share — runs once the dated template is filled
The template ships blank on purpose. After a single-date pull across all 15 peers, this computes & charts the share.

In [ ]:
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
tot=pd.to_numeric(tmpl['Total open roles'],errors='coerce')
ai =pd.to_numeric(tmpl['AI-ML'],errors='coerce')
if tot.notna().any() and (tot.fillna(0)>0).any():
    tmpl['share']=ai/tot; t2=tmpl.dropna(subset=['share']).sort_values('share')
    plt.figure(figsize=(8,5)); plt.barh(t2['Company'],t2['share'])
    plt.xlabel('AI-ML share of open roles'); plt.title('R&D intensity proxy (when filled)')
    plt.tight_layout(); plt.savefig('week02_ai_ml_share.png',dpi=120); print('chart saved')
else:
    print('Template not yet filled. Do a single-date pull across all 15 peers, enter counts, re-run.')

## 5. Key takeaway
Headcount/funding and defensible IP are **decoupled**: Figure AI is well-staffed and extremely well-funded but IP-light (PatentVest), while UBTECH (large team + 3,000+ patents) and Boston Dynamics (large team + deepest legged-robotics moat) lead on combined scale+IP. Benchmark peers on IP + deployments, not raw size or raise. See `executive_summary.pdf`.